# Pravaas — Lab 4: CNN and Transfer Learning

Reproducible railway fault classification. Run from a checkout of this rebuilt repository. The application ships recorded experiments; running this notebook creates a new local experiment. No outputs are fabricated.

**Questions:** How do a scratch CNN, frozen ImageNet features, and fine-tuning differ in performance, training time, and trainable parameters?

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
os.chdir(root)
assert (root / 'ml/train.py').exists(), 'Open this notebook from the rebuilt Pravaas checkout.'
print(root)

## 1. Environment
Install CPU PyTorch first on CPU machines (see README). On Colab, the preinstalled GPU PyTorch can be used. For exact recorded versions see `requirements-lock.txt`.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)

## 2. Dataset and partition audit
The download is approximately 2 GB. Source partitions are preserved, exact decoded-image duplicates removed, and hashes retained. Near-duplicate capture sessions remain a limitation.

In [ ]:
subprocess.run([sys.executable, 'ml/download.py', '--output', 'data/kaggle.zip'], check=True)
subprocess.run([sys.executable, 'ml/prepare.py', 'data/kaggle.zip', '--output', 'data/classification'], check=True)
manifest = json.loads(Path('data/classification/manifest.json').read_text())
print(manifest['counts'], 'duplicates removed:', manifest['duplicates_removed'])

## 3. CNN architecture
Inspect the convolution, ReLU, pooling, dropout, and dense layers. The transfer-learning backbone is MobileNetV3 Small.

In [ ]:
sys.path.insert(0, str(root / 'ml'))
from train import SmallCNN
model = SmallCNN()
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

## 4. Train and select on validation loss
Train all three strategies on the same partitions. Set `--device cuda` for an available CUDA GPU. Keep the test split out of hyperparameter selection.

In [ ]:
subprocess.run([sys.executable, 'ml/train.py', '--data', 'data/classification', '--output', 'artifacts', '--epochs', '12', '--finetune-epochs', '5', '--threads', '2'], check=True)

## 5. Compare actual held-out results
Defective is the positive class. Fine-tuning time is incremental after frozen training. Interpret scores in light of the small test set.

In [ ]:
results = json.loads(Path('artifacts/experiments.json').read_text())
for m in results['models']:
    print(m['name'], {k: round(m[k], 4) for k in ['accuracy','precision','recall','f1','training_seconds']})

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, m in zip(axes, results['models']):
    ax.imshow(m['confusion_matrix'], cmap='Blues')
    for r in range(2):
        for c in range(2): ax.text(c, r, m['confusion_matrix'][r][c], ha='center', va='center')
    ax.set_title(m['name']); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_xticks([0,1], ['Defective','Non-defective']); ax.set_yticks([0,1], ['Defective','Non-defective'])
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for m in results['models']:
    h = m['history']
    axes[0].plot([x['epoch'] for x in h], [x['val_accuracy'] for x in h], label=m['name'])
    axes[1].plot([x['epoch'] for x in h], [x['val_loss'] for x in h], label=m['name'])
axes[0].set_ylabel('Validation accuracy'); axes[1].set_ylabel('Validation loss')
for ax in axes: ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(alpha=.2)
plt.tight_layout()

## 6. Discuss trade-offs
- Compare false negatives with false positives. What does the test size imply about uncertainty?
- Does fine-tuning improve validation loss and test recall? Do not assume it must.
- Compare trainable parameter counts, training duration and accuracy.
- Why are detector mAP and image classification accuracy different metrics?
- What additional data collection and capture-group validation would be needed for practical use?

## 7. Export to the browser workspace
This updates only the local static app. Reload the local page to use the new models; deployment is a separate step.

In [ ]:
subprocess.run([sys.executable, 'ml/publish_artifacts.py', '--artifacts', 'artifacts', '--data', 'data/classification'], check=True)
subprocess.run([sys.executable, 'ml/verify_exports.py'], check=True)